# Module A Report: ACID Validation of the custom B+ Tree

Repo Link: https://github.com/nilay-v3rma/CS432_Assignment_3

Video explanation: https://drive.google.com/file/d/1Pv05nNCXPN0Z8t3cZFrhmn3XffvuUEni/view?usp=sharing 

## 1. Overview
This report documents how Assignment 3 was implemented on top of the Assignment 2 mini-database system.

The existing system already provided:
- A custom database manager
- A table abstraction
- B+ Tree storage per relation

For Assignment 3, transaction management, failure recovery, and concurrency handling were added, then validated through controlled experiments in `Module A/database/test_module_a.py`.

---

## 2. System Architecture Used in Module A
The following new components were used:

- `database/wal.py`
  - Write-Ahead Log (WAL)
  - Appends JSON log records and forces flush (`flush` + `fsync`) for durability

- `database/transaction_manager.py`
  - Transaction lifecycle (`BEGIN`, `COMMIT`, `ROLLBACK`)
  - Tracks operation history for rollback
  - Integrates lock acquisition and lock release

- `database/lock_manager.py`
  - Row-level exclusive lock manager
  - Prevents concurrent write conflicts on same `(table, key)`

- `database/recovery.py`
  - Startup recovery by reading WAL
  - Replays only committed transactions
  - Skips uncommitted transactions 

- `database/test_module_a.py`
  - End-to-end test driver used to validate Atomicity, Consistency, Isolation, and Durability

---

## 3. How Correctness of Operations is Ensured
Correctness was enforced through multiple layers:

### 3.1 Schema correctness
- `Table.validate_record` checks:
  - all required fields exist
  - field types match schema
  - no unknown fields are present

This prevents malformed records from entering B+ Trees.

### 3.2 Transactional operation correctness
- Table operations pass through transaction context (`tx`) during tests.
- Before changing B+ Tree state, each change is registered and written to WAL.
- For rollback, transaction manager replays inverse operations in reverse order:
  - `INSERT` -> delete inserted record
  - `DELETE` -> restore old record
  - `UPDATE` -> restore old value

This ensures all-or-nothing behavior for multi-step workflows.

### 3.3 Application-level consistency rule
A business rule was added for consistency testing:
- A QR code can be issued only if `Person_ID` exists in `Person_Info`.

In test Phase 2B, an invalid QR insert for non-existing `Person_ID` is attempted, raises an exception, and is rolled back. This confirms invalid state is rejected.

---

## 4. How Failures are Handled
Failure handling is based on WAL + rollback + recovery.

### 4.1 During transaction runtime
If an exception happens before commit:
- `rollback()` is called
- inverse operations are applied in reverse order
- transaction is marked `ROLLBACK` in WAL
- locks are released

### 4.2 Crash / restart handling
On restart:
- a new in-memory DB manager and tables are created
- recovery reads WAL from disk
- only transactions with `COMMIT` are replayed
- transactions without commit are ignored

Because storage here is in-memory B+ Trees, uncommitted in-memory state is lost on crash and is not replayed, which effectively preserves atomicity for incomplete transactions.

---

## 5. How Multi-User Conflicts are Handled
A lock manager handles concurrent access conflicts:

- Lock granularity: row-level lock key `(table_name, key)`
- Lock type: exclusive write lock
- Behavior:
  - if lock is free, transaction acquires it
  - if lock is held by another transaction, caller waits (blocking loop)
  - all locks are released on commit/rollback

This prevents concurrent conflicting writes and avoids race-condition corruption.

---

## 6. Experiments Performed
All experiments were executed through `database/test_module_a.py`.

### Phase 1: Initialization
- Database and 4 relations created: `GuestRequest`, `Person_Info`, `QR_Code`, `Log`
- Initial request inserted inside committed transaction

### Phase 2A: Successful multi-relation transaction
Single transaction performs:
1. update guest request
2. insert person profile
3. issue QR code
4. insert log entry

Result: committed successfully.

### Phase 2B: Consistency violation test
- Attempted to issue QR for non-existing person (`P-404`)
- Business rule raised error
- transaction rolled back
- invalid QR record not present afterward

### Phase 3: Atomicity rollback test
- Mid-transaction failure simulated using exception
- partial updates rolled back
- verification confirms no partial records remain

### Phase 4: Isolation / concurrency test
- Two threads update same `GuestRequest` key
- second thread waits until first thread commits
- wait time output confirms lock contention handling

### Phase 5 + 6: Crash-without-commit recovery test
- Start transaction and apply updates
- simulate crash before commit/rollback
- reboot and run recovery
- committed records survive
- crash-time uncommitted records are absent after recovery

---

## 7. Observations
- WAL with forced disk sync provided reliable durability of transaction logs.
- Reverse-operation rollback worked correctly for partial failure paths.
- Lock manager successfully serialized conflicting writes to same row.
- Recovery replay restored committed state after simulated reboot.
- Multi-table transaction behavior matched Assignment 3 goals.

---

## 8. Limitations
Current implementation is correct for assignment scope, but has practical limitations:

1. In-memory data engine
- B+ Trees are reconstructed at startup from WAL replay.
- No direct on-disk page storage for B+ Trees.

2. Concurrency model
- only exclusive row-level lock is implemented
- no shared locks / lock upgrades / deadlock detection

3. Recovery model
- redo of committed transactions only
- no advanced checkpointing or log truncation strategy in tests

4. Performance
- lock waiting uses polling (`sleep`) and may be inefficient under high contention

---

## 9. Conclusion
The Module A implementation satisfies ACID validation requirements:
- Atomicity: demonstrated through rollback and crash-without-commit behavior
- Consistency: demonstrated through schema validation and invalid entry rejection
- Isolation: demonstrated using row-level locking in concurrent updates
- Durability: demonstrated via WAL persistence and post-restart recovery

Overall, the system moved from a functional B+ Tree database prototype (Assignment 2) to a reliable transaction-aware prototype (Assignment 3).